# Chess Transformer — Colab (Elite + masked-CE, resumible)
Flujo válido: setup → secrets → datos → preprocess → piloto → full → eval/paper → artefactos.
Si la sesión muere, re-ejecuta desde la celda de train: continúa desde `checkpoints/<run_id>/last.pt`.


In [ ]:
!git clone https://github.com/oscar2697/chess-transformer.git 2>/dev/null; true
%cd /content/chess-transformer
!git pull origin main && git log --oneline -1


In [ ]:
!pip install python-chess torch matplotlib langchain-openai plotly -q
from src.graph_agents.llm_agents import get_llm
print('import ok, backend:', get_llm()[0])


In [ ]:
from google.colab import userdata
import os
os.environ['LLM_PROVIDER'] = 'nvidia'
os.environ['NVIDIA_API_KEY'] = userdata.get('NVIDIA_KEY')
os.environ['LLM_MODEL'] = 'moonshotai/kimi-k3'
from src.graph_agents.llm_agents import get_llm
print('backend:', get_llm()[0])  # esperado: nvidia


In [ ]:
!mkdir -p data/raw
!wget -q https://database.nikonoel.fr/lichess_elite_2024-01.zip -O data/raw/elite_2024-01.zip
!wget -q https://database.nikonoel.fr/lichess_elite_2024-02.zip -O data/raw/elite_2024-02.zip
!wget -q https://database.nikonoel.fr/lichess_elite_2024-03.zip -O data/raw/elite_2024-03.zip
!unzip -o -q data/raw/elite_2024-01.zip -d data/raw/
!unzip -o -q data/raw/elite_2024-02.zip -d data/raw/
!unzip -o -q data/raw/elite_2024-03.zip -d data/raw/
!ls -lh data/raw/*.pgn


In [ ]:
from src.graph_agents.llm_agents import run_agent
r = run_agent('data_engineer', 'Preprocess 3 elite months',
              overrides={'pgn_path': 'data/raw', 'elo_threshold': 2000,
                         'max_positions': 600000},
              auto=True)
print('mode:', r['mode'])
print(r['result'])


In [ ]:
# PILOTO (3 epocas). Criterio: CE<2.5 y top1 subiendo -> lanzar full.
from src.graph_agents.llm_agents import run_agent
r = run_agent('trainer', 'Masked-CE pilot',
              overrides={'epochs': 3, 'batch_size': 128, 'lr': 1e-3,
                         'warmup_ratio': 0.05, 'mask_illegal': True,
                         'run_id': 'masked-v1', 'seed': 42,
                         'resume': True, 'use_amp': True},
              auto=True)
print('mode:', r['mode'])
print(r['result'])


In [ ]:
# FULL (continua desde el piloto via resume, mismo run_id). Re-ejecutable si muere.
from src.graph_agents.llm_agents import run_agent
r = run_agent('trainer', 'Masked-CE full',
              overrides={'epochs': 15, 'batch_size': 128, 'lr': 1e-3,
                         'warmup_ratio': 0.05, 'mask_illegal': True,
                         'run_id': 'masked-v1', 'seed': 42,
                         'resume': True, 'use_amp': True},
              auto=True)
print('mode:', r['mode'])
print(r['result'])


In [ ]:
from src.graph_agents.llm_agents import run_agent
print(run_agent('evaluator', 'Evaluate policy/value vs Stockfish', auto=True)['result'])
print(run_agent('writer', 'Write results tables', auto=True)['result'][:300])


In [ ]:
!tar -czf /content/artifacts_masked.tar.gz -C /content/chess-transformer checkpoints/masked-v1/ experiments/training_log_masked-v1.jsonl experiments/training_results_masked-v1.json experiments/evaluation_results.json paper/main.tex data/processed/stats.json data/processed/vocab.json
!ls -lh /content/artifacts_masked.tar.gz
from google.colab import files
files.download('/content/artifacts_masked.tar.gz')
